In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import uniform, randint

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer

import pickle
import os

In [2]:
target_train = pd.read_parquet("../data/target_train.parquet")
network_train = pd.read_parquet("../data/network_train.parquet")
weather_train = pd.read_parquet("../data/weather_train.parquet")
weather_test = pd.read_parquet("../data/weather_test.parquet")
network_test = pd.read_parquet("../data/network_test.parquet")

In [3]:
def flatten_weather_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = ["_".join([str(i) for i in col]) for col in df.columns]
    return df

def interpolate_weather(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_index()
    
    df = df.interpolate(method="time")
    df = df.ffill()
    return df

def aggregate_weather(df: pd.DataFrame) -> pd.DataFrame:
    ssrd_cols = [c for c in df.columns if c.endswith("ssrd")]
    tcc_cols  = [c for c in df.columns if c.endswith("tcc")]
    temp_cols = [c for c in df.columns if c.endswith("2t")]
    wind_cols = [c for c in df.columns if c.endswith("100ws")]

    flat = pd.DataFrame(index=df.index)
    flat["ssrd_mean"] = df[ssrd_cols].mean(axis=1)
    flat["ssrd_std"]  = df[ssrd_cols].std(axis=1)
    flat["tcc_mean"]  = df[tcc_cols].mean(axis=1)
    flat["tcc_std"]   = df[tcc_cols].std(axis=1)
    flat["temp_mean"] = df[temp_cols].mean(axis=1)
    flat["temp_std"]  = df[temp_cols].std(axis=1)
    flat["wind_mean"] = df[wind_cols].mean(axis=1)
    flat["wind_std"]  = df[wind_cols].std(axis=1)

    return flat.bfill().ffill()

def prepare_weather(df: pd.DataFrame) -> pd.DataFrame:
    """Flatten multi-index columns, interpolate, and aggregate to per-hour stats."""
    df = flatten_weather_columns(df)
    df = interpolate_weather(df)
    df = aggregate_weather(df)
    df = df.reset_index().rename(columns={df.index.name or "index": "ds"})
    return df

def add_lag_features_load(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["temp_lag1"]  = df["temp_mean"].shift(1)
    df["temp_lag24"] = df["temp_mean"].shift(24)

    return df

def add_rolling_features_load(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Temperature rolling (
    df["temp_roll_3h"]     = df["temp_mean"].rolling(3).mean()
    df["temp_roll_6h"]     = df["temp_mean"].rolling(6).mean()
    df["temp_roll_24h"]    = df["temp_mean"].rolling(24).mean()
    df["temp_roll_3h_std"] = df["temp_mean"].rolling(3).std()
    return df

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt = pd.to_datetime(df["ds"])

    hour = dt.dt.hour
    dayofweek = dt.dt.dayofweek
    month = dt.dt.month

    # Cyclical encoding
    df["hour_sin"]  = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * hour / 24)
    df["dow_sin"]   = np.sin(2 * np.pi * dayofweek / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * dayofweek / 7)
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)

    return df
def engineer_weather_features_load(df: pd.DataFrame) -> pd.DataFrame:
    """Complete feature engineering pipeline FOR LOAD."""
    df = add_lag_features_load(df)
    df = add_rolling_features_load(df)
    df = add_time_features_load(df)
    return df.dropna()


 Preparing weather data- I will use only Temperature

In [4]:
weather_processed = prepare_weather(weather_train)
weather_load = weather_processed[['ds', 'temp_mean', 'temp_std']].copy()

In [5]:
weather_load.head()

,ds,temp_mean,temp_std
0,2020-01-01 00:00:00+00:00,3.114028,3.679529
1,2020-01-01 01:00:00+00:00,2.807910,3.637566
2,2020-01-01 02:00:00+00:00,2.749704,3.592852
3,2020-01-01 03:00:00+00:00,2.656238,3.711994
4,2020-01-01 04:00:00+00:00,2.599818,3.779075


In [6]:
weather_load.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43849 entries, 0 to 43848
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype              
---  ------     --------------  -----              
 0   ds         43849 non-null  datetime64[ns, UTC]
 1   temp_mean  43849 non-null  float32            
 2   temp_std   43849 non-null  float32            
dtypes: datetime64[ns, UTC](1), float32(2)
memory usage: 685.3 KB


ENGINEER FEATURES FOR LOAD

In [7]:

# Engineer features
weather_load = add_lag_features_load(weather_load)
weather_load = add_rolling_features_load(weather_load)
weather_load = add_time_features(weather_load)
weather_load = weather_load.dropna()

print(f"  Weather features: {weather_load.shape}")
print(f"  Columns: {weather_load.columns.tolist()}")



  Weather features: (43825, 15)
  Columns: ['ds', 'temp_mean', 'temp_std', 'temp_lag1', 'temp_lag24', 'temp_roll_3h', 'temp_roll_6h', 'temp_roll_24h', 'temp_roll_3h_std', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']


CREATE LOAD DATAFRAME

In [8]:

weather_load['ds'] = pd.to_datetime(weather_load['ds'])
weather_load = weather_load.set_index('ds')

# Combine target + weather + network
load_df = target_train[['FR_load_actual']].join(weather_load)
load_df = load_df.join(network_train)
load_df = load_df.dropna()

print(f"  Load dataframe shape: {load_df.shape}")
print(f"  Total features: {load_df.shape[1] - 1}")



  Load dataframe shape: (43761, 24)
  Total features: 23


In [9]:
cols_to_drop = ['EEX_CARBON', 'FR_capacity_solar', 'FR_capacity_wind']
load_df = load_df.drop(columns=[c for c in cols_to_drop if c in load_df.columns])


Electricity load/demand is driven by consumer behavior (temperature, time of day, week, season), not by how much generation capacity exists. This is why I dropped the two capacity features. Moreover carbon prices do not influence how much electricity consumers draw.

SPLIT DATA

In [10]:

X_train = load_df.loc['2020':'2023'].drop('FR_load_actual', axis=1)
y_train = load_df.loc['2020':'2023']['FR_load_actual']

X_valid = load_df.loc['2024'].drop('FR_load_actual', axis=1)
y_valid = load_df.loc['2024']['FR_load_actual']

In [11]:
print('\nX_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('X_valid shape:', X_valid.shape)
print('y_valid shape:', y_valid.shape)


X_train shape: (35011, 20)
y_train shape: (35011,)
X_valid shape: (8749, 20)
y_valid shape: (8749,)


In [12]:
X_train.to_parquet('../data/X_train_load.parquet')
X_valid.to_parquet('../data/X_valid_load.parquet')
y_train.to_frame().to_parquet('../data/y_train_load.parquet')
y_valid.to_frame().to_parquet('../data/y_valid_load.parquet')
print("Load features saved.")

Load features saved.


## MODELS

XGBOOST

In [13]:

model_load = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=10,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    tree_method='hist',
    verbosity=0
)

model_load.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [14]:


y_pred_train = model_load.predict(X_train)
y_pred_valid = model_load.predict(X_valid)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
mae_train = mean_absolute_error(y_train, y_pred_train)
wmape_train = np.sum(np.abs(y_train - y_pred_train)) / np.sum(np.abs(y_train))
mape_train = mean_absolute_percentage_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)

rmse_valid = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
mae_valid = mean_absolute_error(y_valid, y_pred_valid)
mape_valid = mean_absolute_percentage_error(y_valid, y_pred_valid)
wmape_valid = np.sum(np.abs(y_valid - y_pred_valid)) / np.sum(np.abs(y_valid))
r2_valid = r2_score(y_valid, y_pred_valid)

print(f"\nTraining: RMSE = {rmse_train:.2f} MW ,MAE: {mae_train:.2f} MW , WMAPE: {wmape_train:.4f} ({wmape_train*100:.2f}%), MAPE: {mape_train:.4f} ({mape_train*100:.2f}%) R²: {r2_train:.4f}")
print(f"\nValidation:")
print(f"  RMSE: {rmse_valid:.2f} MW")
print(f"  MAE: {mae_valid:.2f} MW")
print(f"  WMAPE: {wmape_valid:.4f} ({wmape_valid*100:.2f}%)")
print(f"  MAPE : {mape_valid:.4f} ({mape_valid*100:.2f}%) ")
print(f"  R²: {r2_valid:.4f}")




Training: RMSE = 1634.27 MW ,MAE: 1217.52 MW , WMAPE: 0.0240 (2.40%), MAPE: 0.0246 (2.46%) R²: 0.9782

Validation:
  RMSE: 2796.45 MW
  MAE: 2006.39 MW
  WMAPE: 0.0410 (4.10%)
  MAPE : 0.0409 (4.09%) 
  R²: 0.9175


The model performs well on both training and validation sets, indicated by high R² values and relatively low error metrics. There is a slight drop in performance from training to validation, which is normal and suggests good generalization without significant overfitting.

CROSS-VALIDATION

In [24]:


tscv = TimeSeriesSplit(n_splits=3)
X_cv = load_df.drop('FR_load_actual', axis=1)
y_cv = load_df['FR_load_actual']

rmse_scorer = make_scorer(
    lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
    greater_is_better=False
)

rmse_scores = -cross_val_score(model_load, X_cv, y_cv, cv=tscv, scoring=rmse_scorer)
r2_scores = cross_val_score(model_load, X_cv, y_cv, cv=tscv, scoring='r2')

print(f"\nCross-Validation:")
print(f"  RMSE per fold: {rmse_scores}")
print(f"  Average RMSE: {rmse_scores.mean():.2f} MW")
print(f"  R² per fold: {r2_scores}")
print(f"  Average R²: {r2_scores.mean():.4f}")


Cross-Validation:
  RMSE per fold: [3257.91436352 3217.87662908 3161.95351009]
  Average RMSE: 3212.58 MW
  R² per fold: [0.91090298 0.89552939 0.89183789]
  Average R²: 0.8994


In [15]:
model_l = pd.Series(
    model_load.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("XGBoost Feature Importances (Top 20):")
print(model_l.head(20))

XGBoost Feature Importances (Top 20):
temp_roll_6h               0.344830
temp_roll_24h              0.303012
month_cos                  0.059444
dow_sin                    0.050333
FR_availability_nuclear    0.049897
hour_cos                   0.042849
FR_availability_coal       0.023606
hour_sin                   0.019092
temp_mean                  0.016967
temp_roll_3h_std           0.015460
FR_availability_gas        0.014348
temp_lag24                 0.013877
temp_lag1                  0.013091
temp_roll_3h               0.007551
month_sin                  0.006561
EEX_GAS_PEG                0.006385
EEX_COAL                   0.004381
FR_availability_hydro      0.004120
dow_cos                    0.003462
temp_std                   0.000733
dtype: float32


Feature importance aligns with economic theory. Temperature dominates Load Prediction. This is logical since heating & cooling drives most electricity demand. Time features correctly identify demand cycles.

In [15]:

print("Saving the model")


with open('load_final.pkl', 'wb') as f:
    pickle.dump(model_load, f)



Saving the model


# Predicting on test dataset



In [16]:
weather_processed = prepare_weather(weather_test)
weather_load = weather_processed[['ds', 'temp_mean', 'temp_std']].copy()

# Engineer features
weather_load = add_lag_features_load(weather_load)
weather_load = add_rolling_features_load(weather_load)
weather_load = add_time_features(weather_load)
weather_load = weather_load.dropna()

print(f"  Weather features: {weather_load.shape}")


  Weather features: (8736, 15)


In [17]:
weather_load = weather_load.reset_index(drop=True)  # drop the old integer index cleanly
weather_load['ds'] = pd.to_datetime(weather_load['ds'])
weather_load = weather_load.set_index('ds')          # ← uncomment this

if 'ds' in network_test.columns:
    network_test['ds'] = pd.to_datetime(network_test['ds'])
    network_test = network_test.set_index('ds')

cols_to_drop = ['EEX_CARBON', 'FR_capacity_solar', 'FR_capacity_wind']
network_test = network_test.drop(columns=[c for c in cols_to_drop if c in network_test.columns])

X_test = weather_load.join(network_test)
X_test = X_test.dropna()




In [18]:
X_test


,temp_mean,temp_std,temp_lag1,temp_lag24,temp_roll_3h,temp_roll_6h,temp_roll_24h,temp_roll_3h_std,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,EEX_COAL,EEX_GAS_PEG,FR_availability_coal,FR_availability_gas,FR_availability_hydro,FR_availability_nuclear
ds,,,,,,,,,,,,,,,,,,,,
2025-01-02 00:00:00+00:00,6.123503,2.876559,6.066291,3.516834,6.107876,6.360512,6.137304,0.036381,0.000000,1.000000,0.433884,-0.900969,5.000000e-01,0.866025,110.938866,48.570000,1757.0,11251.370117,13004.0,53053.468750
2025-01-02 01:00:00+00:00,6.088181,2.853961,6.123503,3.447916,6.092659,6.234478,6.247315,0.028867,0.258819,0.965926,0.433884,-0.900969,5.000000e-01,0.866025,110.938866,48.570000,1757.0,11251.370117,13004.0,53053.468750
2025-01-02 02:00:00+00:00,5.914766,2.921683,6.088181,3.418089,6.042150,6.118796,6.351344,0.111722,0.500000,0.866025,0.433884,-0.900969,5.000000e-01,0.866025,110.938866,48.570000,1757.0,11251.370117,13004.0,53053.468750
2025-01-02 03:00:00+00:00,5.666475,2.897085,5.914766,3.341538,5.889808,5.998842,6.448216,0.211958,0.707107,0.707107,0.433884,-0.900969,5.000000e-01,0.866025,110.938866,48.570000,1757.0,11251.370117,13004.0,53053.468750
2025-01-02 04:00:00+00:00,5.427680,2.809566,5.666475,3.327906,5.669641,5.881150,6.535707,0.243558,0.866025,0.500000,0.433884,-0.900969,5.000000e-01,0.866025,110.938866,48.570000,1757.0,11251.370117,13004.0,53053.468750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 19:00:00+00:00,0.128238,3.175300,0.631883,1.762812,0.587312,1.898967,0.758416,0.438491,-0.965926,0.258819,0.974928,-0.222521,-2.449294e-16,1.000000,82.039536,27.108999,1757.0,12928.110352,13294.0,56157.078125
2025-12-31 20:00:00+00:00,-0.226904,3.120010,0.128238,1.432323,0.177739,1.166401,0.689281,0.431528,-0.866025,0.500000,0.974928,-0.222521,-2.449294e-16,1.000000,82.039536,27.108999,1757.0,12928.110352,13294.0,56157.078125
2025-12-31 21:00:00+00:00,-0.485759,3.055300,-0.226904,1.085966,-0.194809,0.495795,0.623793,0.308255,-0.707107,0.707107,0.974928,-0.222521,-2.449294e-16,1.000000,82.039536,27.108999,1757.0,12928.110352,13294.0,56157.078125


In [19]:
y_pred_test = model_load.predict(X_test)
y_pred_test = pd.Series(y_pred_test, index=X_test.index, name="FR_load_predicted")
print(y_pred_test.shape)
y_pred_test.head()

(8736,)


ds
2025-01-02 00:00:00+00:00    61119.945312
2025-01-02 01:00:00+00:00    59468.085938
2025-01-02 02:00:00+00:00    58064.757812
2025-01-02 03:00:00+00:00    58548.027344
2025-01-02 04:00:00+00:00    59823.867188
Name: FR_load_predicted, dtype: float32

In [20]:
os.makedirs("../predictions", exist_ok=True)
X_test.to_parquet('../data/X_test_load.parquet')
l_predictions = y_pred_test.reset_index()
l_predictions.columns = ["datetime", "FR_load_predicted"]
l_predictions.to_parquet("../predictions/FR_load_predicted_2025.parquet", index=False)

print(l_predictions.shape)
l_predictions.head()

(8736, 2)


,datetime,FR_load_predicted
0,2025-01-02 00:00:00+00:00,61119.945312
1,2025-01-02 01:00:00+00:00,59468.085938
2,2025-01-02 02:00:00+00:00,58064.757812
3,2025-01-02 03:00:00+00:00,58548.027344
4,2025-01-02 04:00:00+00:00,59823.867188
